*********************************************************************************************************
*************************************Transformation_Steps*************************************.
1) Standardize & Correct Columns Names, If Needed According To Sneak Case.
2) Standardize Data Types & Better Data Types Leads To Better Performance.
3) Handle Missing Values.
4) Handle Duplicates.
5) Clean & Standardize Text Columns.
6) Clean & Standardize Numeric Columns.
7) Clean & Standardize Date Columns.
8) Fill Missing Values If Needed According The Business Rule.
9) Order Columns Based On Similarity (ID, Numeric, Dates, String).
10) Check The Most important constraint In The Table (Data Integrity) Befor Storing.
11) Store The Transformed Data Into Folder <clean_data>.
*********************************************************************************************************

In [56]:
# Import data analysis libraries.
import pandas as pd
import numpy as np
import os

# Import data profiling and reporting library.
from data_profiling import ProfileReport

# Set  displaying settings.
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 15)

# Paths of raw and clean data folders.
raw_path = r"../datasets/01_bronze_raw_data/" 
clean_path = r"../datasets/02_silver_clean_data/"
report_path = r"../data_quality_reports/"
os.makedirs(report_path, exist_ok=True)

In [57]:
# Create a counter to count number of bytes before changing data types (IN BYTES).
total_memory_before = 0

# Create a counter to count number of bytes after changing data types (IN BYTES).
total_memory_after = 0

*********************************************************************************************************
First: orders Table.

In [58]:
# Read the table.
orders = pd.read_csv(f"{raw_path}olist_orders_dataset.csv", engine="pyarrow")

# Check the shape.
orders.shape  # (112372, 8)

# First: Discover [orders table].
# ** WHAT DOES EACH ROW MEAN? **
# EACH ROW REPRESENTS AN ORDER PLACED BY A CUSTOMER.

(112372, 8)

In [59]:
# Report about orders table.
orders_report = ProfileReport(
    df=orders,
    title="Orders Table Report",
    explorative=True
)

orders_report.to_file(f"{report_path}01_orders_report.html") # The report file will be in the same path of project file.

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:01<00:00,  4.45it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [60]:
orders.dtypes # All data types are objects.

# Calculate how many bytes are used before changing data types.
orders_bytes_before = orders.memory_usage(deep=True).sum() 
total_memory_before += orders_bytes_before

# (order_status) column has repeated values.
orders["order_status"].unique()

# Change data types.
orders = orders.astype({
    "order_id": "string[pyarrow]",
    "customer_id": "string[pyarrow]",
    "order_status": "category",
    "order_purchase_timestamp": "datetime64[s]",
    "order_approved_at": "datetime64[s]",
    "order_delivered_carrier_date": "datetime64[s]",
    "order_delivered_customer_date": "datetime64[s]",
    "order_estimated_delivery_date": "date32[pyarrow]",
})

# Calculate how many bytes are used after changing data types.
orders_bytes_after = orders.memory_usage(deep=True).sum()
total_memory_after += orders_bytes_after

In [61]:
# check missing values.
# [order_approved_at, order_delivered_carrier_date, order_delivered_customer_date] just have missing values.
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  15
order_delivered_carrier_date     1184
order_delivered_customer_date    2360
order_estimated_delivery_date       0
dtype: int64

In [62]:
# Check duplicates.
orders.duplicated().sum() # There is 14455 duplicated rows.

# Drop Duplicates.
orders.drop_duplicates(inplace=True)

In [63]:
# Check data quality issues for string columns.

# Ensure that there is no any value in ID columns out of this range A-Za-z0-9 and,
# for order_status range A-Za-z 
assert orders["order_id"].str.fullmatch(pat=r"[a-z0-9]+", na=False).all()
assert orders["customer_id"].str.fullmatch(pat=r"[a-z0-9]+", na=False).all()
assert orders["order_status"].str.fullmatch(pat=r"[a-z]+", na=True).all()

In [64]:
# Check data quality issues for data columns.
date_cols = ["order_purchase_timestamp",
             "order_approved_at",
             "order_delivered_carrier_date",
             "order_delivered_customer_date",
             "order_estimated_delivery_date"]

# Ensure there is no any invalid dates (dates out of the range from 2016 to 2018).
for date_col in date_cols:
    if ((orders[date_col].dt.year > 2018) | (orders[date_col].dt.year < 2016)).any():
        print(f"{date_col} has invalid date!")

# Ensure the logical correctness of series operations.
# Ensure the logical correctness of series operations.
(orders["order_purchase_timestamp"] > orders["order_approved_at"]).sum()
(orders["order_approved_at"] > orders["order_delivered_carrier_date"]).sum() # has a problem.
(orders["order_delivered_carrier_date"] > orders["order_delivered_customer_date"]).sum() # has a problem.


orders["order_delivered_carrier_date"] = np.where(
    orders["order_approved_at"] > orders["order_delivered_carrier_date"],
    orders["order_approved_at"],
    orders["order_delivered_carrier_date"]
)

orders["order_delivered_customer_date"] = np.where(
    orders["order_delivered_carrier_date"] > orders["order_delivered_customer_date"],
    orders["order_delivered_carrier_date"],
    orders["order_delivered_customer_date"]
)

In [65]:
# Reorder columns according to the rule.
orders = orders[["order_id", "customer_id",
                 "order_purchase_timestamp", "order_approved_at",
                 "order_delivered_carrier_date", "order_delivered_customer_date",
                 "order_estimated_delivery_date", "order_status"]]

In [66]:
# We will use parquet format to keep the the new data types as they are.
orders.to_parquet(f"{clean_path}orders.parquet", index=False, engine="pyarrow")

###########################################################################################################
Second: customers Table.

In [67]:
# Read the table.
customers = pd.read_csv(f"{raw_path}olist_customers_dataset.csv", engine="pyarrow")

# Check the shape.
customers.shape # (99441, 5)

# Second: [customers table].
# ** WHAT DOES EACH ROW MEAN? **
# EACH ROW REPRESENTS A CUSTOMER INFORMATION WITH CUSTOMER_ID COLUMN REPEATS THE 
# INFORMATION EACH NEW TRANSACTION.

(99441, 4)

In [68]:
# profile report about customers table.
customers_report = ProfileReport(
    df=customers,
    title="Customers Table Report",
    explorative=True
)

customers_report.to_file(f"{report_path}02_customers_report.html") # The report file will be in the same path of this python file.

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:01<00:00,  2.12it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [69]:
# Check data types.
customers.dtypes # All data types are objects.

# How many Bytes are used befor changing data types.
customers_bytes_before = customers.memory_usage(deep=True).sum()
total_memory_before += customers_bytes_before
# total_memory_before += customers_bytes_before

# Change data types and save memory as much as possible.
customers = customers.astype({
    "customer_id": "string[pyarrow]",
    "customer_unique_id": "string[pyarrow]",
    "customer_city": "category",
    "customer_state": "category"
    
})

# # How many bytes are used after changing data type.
customers_bytes_after = customers.memory_usage(deep=True).sum()
total_memory_after += customers_bytes_after

In [70]:
# Check missing values.
customers.isna().sum() # There is no any mising values in the whole table.

customer_id           0
customer_unique_id    0
customer_city         0
customer_state        0
dtype: int64

In [71]:
# Check duplicates.
customers.duplicated().sum() # There is no any duplicates.
customers["customer_id"].duplicated().sum() # There is no any duplicate.

np.int64(0)

In [72]:
# Check data quality issues for string columns.
assert customers["customer_id"].str.fullmatch(pat=r"[a-z0-9]+", na=False).all()
assert customers["customer_unique_id"].str.fullmatch(pat=r"[a-z0-9]+", na=False).all()
customers["customer_city"] = customers["customer_city"].str.strip()
customers["customer_state"] = customers["customer_state"].str.strip()

In [73]:
# Reorder the columns according to the rule.
customers = customers[["customer_unique_id",
                       "customer_id",
                       "customer_city",
                       "customer_state"]]

In [74]:
# We will use parquet format to keep the the new data types as they are.
customers.to_parquet(f"{clean_path}customers.parquet", index=False,engine="pyarrow")

#######################################################################################################
Third: order_items table.

In [75]:
# Read and inspect the order_items table.
order_items = pd.read_csv(f"{raw_path}olist_order_items_dataset.csv", engine="pyarrow")

# Check the shape.
order_items.shape # (156424, 7).

# Third: [order_items table].
# ** WHAT DOES EACH ROW MEAN? **
# EACH ROW REPRESENTING A SINGLE ITEM WITHIN AN ORDER.

(156424, 7)

In [76]:
# Report about order_items table.
order_items_report = ProfileReport(
    df=order_items,
    title="Order Items Table report",
    explorative=True
)

order_items_report.to_file(f"{report_path}03_order_items_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:02<00:00,  3.14it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [77]:
# Check data types.
order_items.dtypes # All data types are objects.

# Memory usage in Bytes befor changing data types.
order_items_bytes_before = order_items.memory_usage(deep=True).sum()
total_memory_before += order_items_bytes_before

# Changing data types.
# Price and freight_value have decimal numbers, so the proper data type is float.
# Order_id, product_id, seller_id columns have repeated values so, the proper data types is category.
order_items = order_items.astype({
        "order_id": "string[pyarrow]",
        "product_id": "string[pyarrow]",
        "seller_id": "string[pyarrow]",
        "shipping_limit_date": "datetime64[s]",
        "order_item_id": pd.Int8Dtype(),
        "price": pd.Float64Dtype(),
        "freight_value": pd.Float64Dtype()
        })

# # Check number of bytes used after changing data types.
order_items_bytes_after = order_items.memory_usage(deep=True).sum()
total_memory_after += order_items_bytes_after

In [78]:
# Check missing values.
order_items.isna().sum() # There is no missing values in the table

order_id               0
product_id             0
order_item_id          0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [79]:
# Check duplicates.
order_items.duplicated().sum() # There is 37171 dupicated rows.
order_items.drop_duplicates(inplace=True)

In [80]:
# Check data quality issues.
assert order_items["order_id"].str.fullmatch(pat=r"[a-z0-9]+", na=False).all()
assert order_items["product_id"].str.fullmatch(pat=r"[a-z0-9]+", na=False).all()
assert order_items["seller_id"].str.fullmatch(pat=r"[a-z0-9]+", na=False).all()

In [81]:
# Check the quality of numeric columns.
assert order_items["price"].min() > 0
assert order_items["freight_value"].min() >= 0 # There might be a free shiping.
assert order_items["shipping_limit_date"].dt.year.max() < 2021
assert (order_items["order_item_id"] >= 1).all()

In [82]:
# Reorder columns according to the rules.
order_items = order_items[["order_id",
                           "product_id",
                           "seller_id",
                           "order_item_id",
                           "price",
                           "freight_value",
                           "shipping_limit_date"]]

In [83]:
# Store the table in parquet format to keep the data types after changing.
order_items.to_parquet(f"{clean_path}order_items.parquet", index=False, engine="pyarrow")

###########################################################################################################
Fourth: order_review table.

In [84]:
# Read and inspect the order_reviews table.
order_reviews = pd.read_csv(f"{raw_path}olist_order_reviews_dataset.csv", engine="pyarrow")

# Check the shape.
order_reviews.shape # (97455, 5)

# First: Discover [order_reviews table].
# ** WHAT DOES EACH ROW MEAN? **
# EACH ROW REPRESENTS A REVIEW FOR AN ORDER.

(97455, 5)

In [85]:
oreder_reviews_report = ProfileReport(
    df=order_reviews,
    title="Order Reviews Table Report",
    explorative=True
)

oreder_reviews_report.to_file(f"{report_path}04_order_reviews_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:01<00:00,  3.13it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [86]:
# Change data types.
order_reviews.dtypes # All data types are objects.

# Memory usage in Byte before changing data types.
order_reviews_bytes_before = order_reviews.memory_usage(deep=True).sum()
total_memory_before += order_reviews_bytes_before

order_reviews = order_reviews.astype({
    "review_id": "string[pyarrow]", 
    "order_id": "string[pyarrow]", 
    "review_score": pd.UInt8Dtype(), 
    "review_creation_date": "datetime64[s]",
    "review_answer_timestamp": "datetime64[s]"
})

# Memory usage in Byte after changing data types.
order_review_bytes_after = order_reviews.memory_usage(deep=True).sum()
total_memory_after += order_review_bytes_after

In [87]:
# check missing values.
order_reviews.isna().sum() # There is no missing values.

review_id                  0
order_id                   0
review_score               0
review_creation_date       0
review_answer_timestamp    0
dtype: int64

In [88]:
# Check duplicates.
order_reviews.duplicated().sum() # There is no any duplicates.

np.int64(0)

In [89]:
# Check data quality issues.

assert order_reviews["review_id"].str.fullmatch(pat=r"[a-z0-9]+", na=False).all()
assert order_reviews["order_id"].str.fullmatch(pat=r"[a-z0-9]+", na=False).all()
assert order_reviews["review_score"].between(1, 5).all()
assert (order_reviews["review_creation_date"] < order_reviews["review_answer_timestamp"]).all()

In [90]:
# Reorder table columns.
order_reviews = order_reviews[["review_id",
                               "order_id",
                               "review_score",
                               "review_creation_date",
                               "review_answer_timestamp"]]

In [91]:
# Store the data in parquet format.
order_reviews.to_parquet(f"{clean_path}order_reviews.parquet", index=False, engine="pyarrow")

##########################################################################################################
Fifth: products table.

In [92]:
# Read and inspect the table.
products = pd.read_csv(f"{raw_path}olist_products_dataset.csv", engine="pyarrow")

# Check the shape.
products.shape # (32951, 4)

# First: Discover [products table].
# ** WHAT DOES EACH ROW MEAN? **
# EACH ROW REPRESENTS A DESCRIPTION FOR A PRODUCT.

(32951, 4)

In [93]:
# Profile report about products table.
products_report = ProfileReport(
    df=products,
    title="Products Table Report",
    explorative=True
)

products_report.to_file(f"{report_path}05_products_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 16.25it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [94]:
# Change data types.
# Ckeck number of Bytes used befor changing data types.
products_bytes_before = products.memory_usage(deep=True).sum()
total_memory_before += products_bytes_before 

products = products.astype({
    "product_id": "string[pyarrow]",
    "product_name_length": pd.Int8Dtype(),
    "product_description_length": pd.Int16Dtype(),
    "product_photos_qty": pd.Float32Dtype()
    })

# # Ckeck number of Bytes used after changing data types.
products_bytes_after = products.memory_usage(deep=True).sum()
total_memory_after += products_bytes_after

In [95]:
# Check duplicates.
products.duplicated().sum() # There is no duplicates.

np.int64(0)

In [96]:
# Check missing values.
products.isna().sum()

# Fill nan in numeric values with 0.
products.fillna(0, inplace=True)

In [97]:
# Check data quality issues.
assert products["product_id"].str.fullmatch(pat=r"[a-z0-9]+", na=False).all()
assert (products["product_name_length"] >= 0).all()
assert (products["product_description_length"] >= 0).all()
assert (products["product_photos_qty"] >= 0).all()

In [98]:
# Reorder the columns according the rule.
products = products[["product_id",
                     "product_name_length",
                     "product_description_length",
                     "product_photos_qty"]]

In [99]:
# Store the data in parquet format.
products.to_parquet(f"{clean_path}products.parquet", index=False, engine="pyarrow")

##########################################################################################################
Sixth: Sellers table.

In [100]:
# Read the table.
sellers = pd.read_csv(f"{raw_path}olist_sellers_dataset.csv", engine="pyarrow")

# Check the shape.
sellers.shape # (3095, 3)

# First: Discover [SELLERS].
# ** WHAT DOES EACH ROW MEAN? **
# EACH ROW REPRESENTS A SINGLE SELLER INFORMATION.

(3095, 3)

In [101]:
# Profile report about sellers table.
sellers_report = ProfileReport(
    df=sellers,
    title="Sellers Table Report",
    explorative=True
)

sellers_report.to_file(f"{report_path}06_sellers_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 46.18it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [102]:
# Change data types.

# How many bytes are used before changing data types.
sellers_bytes_before = sellers.memory_usage(deep=True).sum()
total_memory_before += sellers_bytes_before


# Change data types.
sellers = sellers.astype({
    "seller_id": "string[pyarrow]",
    "seller_city": "category",
    "seller_state": "category"
})

# How many bytes are used after changing data types.
sellers_bytes_after = sellers.memory_usage(deep=True).sum()
sellers_bytes_after
total_memory_after += sellers_bytes_after

In [103]:
# Check missing values.
sellers.isna().sum() # There is no any missing values.

seller_id       0
seller_city     0
seller_state    0
dtype: int64

In [104]:
# Check duplicates.
sellers.duplicated().sum()
sellers["seller_id"].duplicated().sum()

np.int64(0)

In [105]:
# Check data quality issues.
assert sellers["seller_id"].str.fullmatch(pat=r"[a-z0-9]+", na=False).all()
assert sellers["seller_state"].str.fullmatch(pat=r"[A-Z]+", na=False).all()

In [106]:
# Reorder the columns.
sellers = sellers[["seller_id",
                   "seller_city",
                   "seller_state"]]

In [107]:
# Store the table in parquet format.
sellers.to_parquet(f"{clean_path}sellers.parquet", index=False, engine="pyarrow")

In [108]:
# Total memory used before changing data types.
total_memory_before

np.int64(121491838)

In [109]:
# Total memory used after changing data types.
total_memory_after

np.int64(56322496)

In [110]:
# Saved bytes.
(total_memory_before - total_memory_after)

np.int64(65169342)